# MiraeVaani 2.0 — Colab #2: LLM Service

Runs **Ollama + Gemma 2 9B** for multilingual Indian language dialog generation.

- OpenAI-compatible API at `POST /v1/chat/completions`
- Responds in the same language the customer speaks
- Exposed via ngrok

**Setup:** Runtime → Change runtime type → **T4 GPU**

In [1]:
# Cell 1: Verify GPU
import subprocess, torch
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip())
print('CUDA:', torch.cuda.is_available())
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

GPU: Tesla T4, 15360 MiB
CUDA: True
VRAM: 15.6 GB


In [2]:
# Cell 2: Install Ollama + ngrok
import subprocess, os, shutil

# zstd needed by Ollama installer
subprocess.run('apt-get install -y -q zstd', shell=True, check=True)

# Install Ollama
r = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    print('❌ Ollama install failed:', r.stderr[-400:])
else:
    os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
    print('✅ Ollama installed:', shutil.which('ollama'))

!pip install -q pyngrok httpx
print('✅ Done')

✅ Ollama installed: /usr/local/bin/ollama
✅ Done


In [3]:
# Cell 3: Configure ngrok
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3G2uzWsQ2TmMqLFaKLwDGccL08C_5VBHnFWrkYftHpVW3bm3v"  # https://dashboard.ngrok.com
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print('✅ ngrok configured')

✅ ngrok configured


In [4]:
# Cell 4: Start Ollama + pull Gemma 2 9B
import subprocess, time, os, httpx

os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
OLLAMA_BIN = '/usr/local/bin/ollama'

if not os.path.exists(OLLAMA_BIN):
    raise FileNotFoundError('Ollama not found — re-run Cell 2')

# Start server
proc = subprocess.Popen(
    [OLLAMA_BIN, 'serve'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env={**os.environ, 'OLLAMA_HOST': '0.0.0.0:11434'},
)

# Wait for API
print('Waiting for Ollama API...')
for i in range(12):
    try:
        if httpx.get('http://localhost:11434/api/tags', timeout=3).status_code == 200:
            print(f'✅ Ollama API ready ({(i+1)*3}s)')
            break
    except Exception:
        pass
    time.sleep(3)
else:
    print('⚠️  Ollama not responding — check install')

# Pull Gemma 2 9B (~5GB)
print('\nPulling Gemma 2 9B... (~5-8 min)')
r = subprocess.run([OLLAMA_BIN, 'pull', 'gemma2:9b'])
print('✅ Gemma 2 9B ready' if r.returncode == 0 else '❌ Pull failed')

Waiting for Ollama API...
✅ Ollama API ready (6s)

Pulling Gemma 2 9B... (~5-8 min)
✅ Gemma 2 9B ready


In [5]:
# Cell 5: Smoke test + expose via ngrok
import httpx
from pyngrok import ngrok

print('Running smoke test (first call loads model into VRAM ~60-120s)...')
try:
    resp = httpx.post(
        'http://localhost:11434/v1/chat/completions',
        json={
            'model': 'gemma2:9b',
            'messages': [{'role': 'user', 'content': 'Say hello in Hindi in one sentence.'}],
            'max_tokens': 50,
        },
        timeout=180,
    )
    print('✅ LLM test:', resp.json()['choices'][0]['message']['content'])
except httpx.ReadTimeout:
    print('⚠️  Smoke test timed out — model is still warming up, proceed anyway')

tunnel = ngrok.connect(11434, 'http')
print()
print('=' * 60)
print('  PASTE THIS INTO YOUR LAPTOP .env')
print('=' * 60)
print(f'LLM_BASE_URL={tunnel.public_url}')
print('=' * 60)

Running smoke test (first call loads model into VRAM ~60-120s)...
✅ LLM test: नमस्ते!  (Namaste!) 


This means "Hello" and is a respectful greeting.  😊


  PASTE THIS INTO YOUR LAPTOP .env
LLM_BASE_URL=https://spud-reusable-carnivore.ngrok-free.dev


In [ ]:
# Cell 6: Keep-alive
import time, httpx
print('LLM keep-alive running...')
i = 0
while True:
    try:
        ok = httpx.get('http://localhost:11434/api/tags', timeout=3).status_code == 200
        i += 1
        if i % 20 == 0:
            print(f'[{i*30}s] LLM alive={ok}')
    except Exception as e:
        print(f'⚠️  {e}')
    time.sleep(30)

LLM keep-alive running...
[600s] LLM alive=True
